Geoinformatik 2 – Web-GIS
# **Übung 3** · Interaktive Webkarte mit Folium & Python
Prof. Dr. Yu Feng


## Lernziele
Nach dieser Übung können Sie:
- mit **Folium** (Leaflet in Python) eine interaktive Karte erzeugen,
- **Punkte**, **Linien** und **Flächen (GeoJSON/Polygon)** visualisieren,
- Popups/Tooltips und LayerControl nutzen,
- eine HTML-Datei `*.html` herunterladen.


## Reference

Getting started — Folium 0.20.0 documentation
- https://python-visualization.github.io/folium/latest/getting_started.html#

## Setup
Führe die Zellen nacheinander aus. Falls du deine Ergebnisse im **Google Drive** speichern willst, mounte Drive.


In [30]:
# Umgebung erkennen und Ausgabeverzeichnis festlegen
from pathlib import Path

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    output_dir = Path("/content")
else:
    # Start im Projektverzeichnis oder im Verzeichnis Uebung3 unterstützen.
    working_dir = Path.cwd()
    exercise_dir = next(
        (directory for parent in (working_dir, *working_dir.parents)
         for directory in (parent, parent / "Uebung3")
         if (directory / "Uebung3_Folium.ipynb").is_file()
         or (directory / "Uebung3_Folium_Student.ipynb").is_file()),
        working_dir,
    )
    output_dir = exercise_dir / "maps"

output_dir.mkdir(parents=True, exist_ok=True)
print("Umgebung:", "Google Colab" if IN_COLAB else "Lokal / Jupyter")
print("Ausgabeverzeichnis:", output_dir.resolve())


def save_map(map_object, filename):
    """Karte speichern und in Google Colab zum Download anbieten."""
    output_file = output_dir / filename
    map_object.save(str(output_file))
    print(f"Gespeichert: {output_file}")
    if IN_COLAB:
        files.download(str(output_file))


Umgebung: Lokal / Jupyter
Ausgabeverzeichnis: C:\Users\dominic_j\Documents\GitHub\Geoinformatik2_HS_Mainz\Uebung3\assets


In [31]:
# Falls Sie lokale Jupyter Notebooks verwenden, installieren Sie die Bibliothek folium mit:
! pip install folium


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1) Grundkarte (Basiskarte)

In [32]:
import folium

# Grundkarte mit OSM-France
m = folium.Map(location=[49.2, 8.5], zoom_start=6, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.fr/osmfr/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors, SRTM | OpenTopoMap",
    name="OSM-France",
    show=False
).add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m

In [33]:
# Karte speichern
save_map(m, "map_1_basiskarte.html")

Gespeichert: c:\Users\dominic_j\Documents\GitHub\Geoinformatik2_HS_Mainz\Uebung3\assets\map_1_basiskarte.html


## 2) Punkte (Marker) mit Popup/Tooltip


Füge mindestens **3 persönliche Punkte** hinzu, jeweils mit Popup und unterschiedlichen Icons/Farben.

In [37]:
import folium
from folium import Popup, Icon

# Grundkarte mit OSM-France
m = folium.Map(location=[49.2, 8.5], zoom_start=6, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.fr/osmfr/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors, SRTM | OpenTopoMap",
    name="OSM-France",
    show=False
).add_to(m)

# Marker: Titel, (lat, lon), Farbe, Icon, Popup-HTML
punkte = [
    ("Mainz", (50.0000, 8.2711), "blue", "book", "<b>Mainz</b><br>Mein Studienort."),
    ("Pfullendorf", (47.924349, 9.255982), "green", "home", "<b>Pfullendorf</b><br>Meine Heimatstadt in Oberschwaben."),
    ("Ramsen (Pfalz)", (49.536667, 8.013056), "red", "flag", "<b>Ramsen (Pfalz)</b><br>Ein Ort in der Pfalz, den ich gut kenne."),
]

bounds = []
for titel, (lat, lon), farbe, icon_name, html in punkte:
    folium.Marker(
        location=[lat, lon],
        tooltip=titel,
        popup=Popup(html, max_width=250),
        icon=Icon(color=farbe, icon=icon_name),
    ).add_to(m)
    bounds.append([lat, lon])

if bounds:
    m.fit_bounds(bounds, padding=(20, 20))

folium.LayerControl(collapsed=False).add_to(m)

m

In [35]:
# Karte speichern
save_map(m, "map_2_punkte.html")

Gespeichert: c:\Users\dominic_j\Documents\GitHub\Geoinformatik2_HS_Mainz\Uebung3\assets\map_2_punkte.html


## 3) Linie (Polyline) – Reiseroute in Sequenz

Erstelle **eine Reiseroute (mind. 3 Stationen)** und prüfe, ob die Nummerierung korrekt erscheint.

In [36]:
import folium
from folium.features import DivIcon

# Grundkarte mit OSM-France
m = folium.Map(location=[51.0, 6.8], zoom_start=5, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.fr/osmfr/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors, SRTM | OpenTopoMap",
    name="OSM-France",
    show=False
).add_to(m)

# Reiseroute (in Reihenfolge)
route = [
    ("Mainz", (50.0000, 8.2711)),
    ("Berlin", (52.5200, 13.4050)),
    ("Den Haag", (52.0705, 4.3007)),
    ("Paris", (48.8566, 2.3522)),
]

# Polyline einzeichnen
folium.PolyLine(
    locations=[coords for _, coords in route],
    color="#0004f9",
    weight=4,
    opacity=0.85,
    tooltip="Reiseroute"
).add_to(m)

# Stationen nummerieren + Marker mit Tooltip/Popup
for i, (name, (lat, lon)) in enumerate(route, start=1):
    folium.Marker(
        location=[lat, lon],
        tooltip=f"{i}. {name}",
        popup=f"<b>{i}. {name}</b><br>Station meiner Reiseroute.",
        icon=DivIcon(html=f"<div style='font-size: 14px; color: white; background: #0004f9; border: 2px solid white; border-radius: 50%; width: 28px; height: 28px; line-height: 24px; text-align: center;'>{i}</div>")
    ).add_to(m)

m.fit_bounds([[lat, lon] for _, (lat, lon) in route], padding=(20, 20))
folium.LayerControl(collapsed=False).add_to(m)
m

IndentationError: expected an indented block after 'for' statement on line 27 (2616171808.py, line 43)

In [ ]:
# Karte speichern
save_map(m, "map_3_reiseroute.html")

## 4) Flächen (Polygon / GeoJSON) – Welt laden & besuchte Länder hervorheben

 Lade ein Welt-GeoJSON (oder nutze den Auto-Download) und hebe besuchte Länder farblich hervor.

In [ ]:
import folium
import requests
from folium import GeoJson

# 1) Welt-GeoJSON ONLINE laden (kein lokaler Download nötig)
geojson_url = "https://raw.githubusercontent.com/johan/world.geo.json/master/countries.geo.json"
response = requests.get(geojson_url, timeout=30)
response.raise_for_status()
geojson_data = response.json()

# 2) Menge der besuchten Länder – Namen müssen zum GeoJSON passen
visited = {
    "Germany", "Switzerland", "Austria", "Luxembourg", "Netherlands",
    "Belgium", "France", "Spain", "Portugal", "United Kingdom",
}

# 3) Karte mit OSM-France anlegen
m = folium.Map(location=[48, 8], zoom_start=4, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.fr/osmfr/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors, SRTM | OpenTopoMap",
    name="OSM-France",
    show=False
).add_to(m)

# 4) Styling-Funktionen
def style_fn(feature):
    props = feature.get("properties", {})
    country = props.get("ADMIN") or props.get("name") or ""
    if country in visited:
        return {"fillColor": "#22c55e", "color": "#065f46", "weight": 1, "fillOpacity": 0.6}
    return {"fillColor": "#94a3b8", "color": "#334155", "weight": 0.5, "fillOpacity": 0.15}

def highlight_fn(_):
    return {"fillOpacity": 0.8, "weight": 2, "color": "#0ea5e9"}

# 5) GeoJSON zur Karte hinzufügen
GeoJson(
    data=geojson_data,
    name="World",
    style_function=style_fn,
    highlight_function=highlight_fn,
    tooltip=folium.features.GeoJsonTooltip(
        # Diese Datei nutzt 'name' → falls nötig auf ["ADMIN"] ändern
        fields=["name"],
        aliases=["Land:"],
        sticky=True
    )
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m

In [ ]:
# Karte speichern
save_map(m, "map_4_flaechen_online.html")

## 5) Checkliste
- [ ] Füge **mindestens 3** persönliche Punkte hinzu, jeweils mit **Popup** und unterschiedlichen Icons/Farben.
- [ ] Erstelle eine **Reiseroute** (mind. 3 Stationen) und prüfe, ob die **Nummerierung** korrekt erscheint.
- [ ] Lade ein Welt-GeoJSON (oder nutze den Auto-Download) und **hebe besuchte Länder** farblich hervor.
- [ ] Exportiere die Karte als **`maps/*.html`** und **lade** die Datei herunter.

> **Download-Hinweis (Colab):** Rechtsklick auf `*.html` im Dateibrowser von Colab → **Download**.

